# 👩‍👦 Autonomous Job-Card Engine (AJE) - Interactive Notebook

Welcome to the **AJE Interactive Playground**! 

This notebook provides a fully self-contained, cell-by-cell walkthrough of the **Autonomous Job-Card Engine** architecture. Instead of relying on passive text, you can execute each component, modify safety rules in real-time, view the self-healing compiler loop, and inspect generated artifacts directly inside this notebook.

### 🏗️ What We Are Building & Testing:
1. **Stateful Cards**: A **Mother Card** (guardian defining safety, code principles, and budget) and **Child Cards** (workers carrying tactical objectives).
2. **Privacy Guard**: A sanitization filter that scrubs secrets (passwords, API keys) and enforces local execution if proprietary paths are accessed.
3. **Sandbox Runner**: An execution space that blocks dangerous commands (like `rm -rf`) and runs verification commands.
4. **Self-Healing Loop**: The model tries to write code, fails a verification check, analyzes the traceback, and corrects its own code automatically.
5. **Autonomous Discovery**: The Mother Card scans the finished code, identifies structural gaps, and generates new child cards autonomously.

---

## 🛠️ Step 1: Environment Setup & YAML Support

In [ ]:
import os
import sys
import shutil
import json
import re
from datetime import datetime
from typing import List, Dict, Any
from dataclasses import dataclass, field

# Ensure PyYAML is installed for card parsing
try:
    import yaml
    print("✅ PyYAML is available!")
except ImportError:
    print("Installing PyYAML...")
    !{sys.executable} -m pip install pyyaml
    import yaml

## 👩‍👦 Step 2: Define Card Models

In [ ]:
@dataclass
class MotherCard:
    id: str
    name: str
    core_philosophy: str
    privacy_mode: str
    restricted_paths: List[str] = field(default_factory=list)
    banned_commands: List[str] = field(default_factory=list)
    global_context: Dict[str, Any] = field(default_factory=dict)
    family_health: str = "Healthy"
    total_spend_usd: float = 0.0
    active_children: List[str] = field(default_factory=list)
    completed_children: List[str] = field(default_factory=list)
    backlog_queue: List[Dict[str, Any]] = field(default_factory=list)

    @classmethod
    def from_dict(cls, data: Dict) -> "MotherCard":
        spec = data.get("spec", {})
        metadata = data.get("metadata", {})
        status = data.get("status", {})
        return cls(
            id=metadata.get("id"),
            name=metadata.get("name"),
            core_philosophy=spec.get("core_philosophy", ""),
            privacy_mode=spec.get("safety_policies", {}).get("privacy_mode", "hybrid"),
            restricted_paths=spec.get("safety_policies", {}).get("restricted_paths", []),
            banned_commands=spec.get("safety_policies", {}).get("banned_commands", []),
            global_context=spec.get("global_context", {}),
            family_health=status.get("family_health", "Healthy"),
            total_spend_usd=status.get("total_spend_usd", 0.0),
            active_children=status.get("active_children", []),
            completed_children=status.get("completed_children", []),
            backlog_queue=status.get("backlog_queue", [])
        )

@dataclass
class ChildCard:
    id: str
    parent_mother_id: str
    name: str
    tactical_objective: str
    deliverables: List[Dict[str, str]] = field(default_factory=list)
    validation_commands: List[str] = field(default_factory=list)
    phase: str = "Pending"
    current_iteration: int = 0
    max_iterations: int = 5
    execution_profile_assigned: str = "local"
    logs: List[str] = field(default_factory=list)

    @classmethod
    def from_dict(cls, data: Dict) -> "ChildCard":
        spec = data.get("spec", {})
        metadata = data.get("metadata", {})
        status = data.get("status", {})
        return cls(
            id=metadata.get("id"),
            parent_mother_id=metadata.get("parent_mother_id"),
            name=metadata.get("name"),
            tactical_objective=spec.get("tactical_objective", ""),
            deliverables=spec.get("deliverables", []),
            validation_commands=spec.get("validation", {}).get("test_commands", []),
            phase=status.get("phase", "Pending"),
            current_iteration=status.get("current_iteration", 0),
            max_iterations=spec.get("max_iterations", 5),
            execution_profile_assigned=status.get("execution_profile_assigned", "local"),
            logs=status.get("logs", [])
        )
print("✅ Card Models successfully defined!")

## 🛡️ Step 3: Implement Privacy Guard and Sandbox Runner

In [ ]:
class PrivacyGuard:
    def __init__(self, restricted_paths: List[str] = None):
        self.restricted_paths = restricted_paths or []
        self.secret_patterns = [
            re.compile(r"(?i)(api[_-]?key|secret|password|passwd|token|bearer|credential|pwd)\s*[:=]\s*['\"][^'\"]+['\"]"),
            re.compile(r"(?i)gho_[a-zA-Z0-9]{36}"),
            re.compile(r"(?i)sk-[a-zA-Z0-9]{48}"),
        ]

    def sanitize_content(self, text: str) -> str:
        sanitized = text
        for pattern in self.secret_patterns:
            def scrub_repl(match):
                match_str = match.group(0)
                parts = re.split(r'([:=])', match_str, maxsplit=1)
                if len(parts) == 3:
                  return f"{parts[0]}{parts[1]} \"[SCRUBBED_BY_PRIVACY_GUARD]\""
                return "\"[SCRUBBED_BY_PRIVACY_GUARD]\""
            sanitized = pattern.sub(scrub_repl, sanitized)
        return sanitized

    def evaluate_routing_profile(self, files_accessed: List[str], base_mode: str) -> str:
        if base_mode == "local":
            return "local"
        for file_path in files_accessed:
            for restricted_pattern in self.restricted_paths:
                regex_pattern = re.escape(restricted_pattern).replace(r"\*", ".*")
                if re.search(regex_pattern, file_path):
                    return "local"
        return "cloud"

import subprocess
class SandboxRunner:
    def __init__(self, banned_commands: List[str] = None):
        self.banned_commands = banned_commands or ["rm -rf", "docker system prune"]

    def is_safe_command(self, cmd: str) -> bool:
        cmd_normalized = cmd.strip().lower()
        for banned in self.banned_commands:
            if banned.strip().lower() in cmd_normalized:
                return False
        return True

    def run_validation(self, cmd: str, cwd: str = ".") -> Dict[str, Any]:
        if not self.is_safe_command(cmd):
            return {
                "success": False,
                "exit_code": -1,
                "output": f"BLOCKED: Command '{cmd}' violated Sandbox Safety Policy!"
            }
        try:
            result = subprocess.run(
                cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=15
            )
            return {
                "success": result.returncode == 0,
                "exit_code": result.returncode,
                "output": result.stdout + "\n" + result.stderr
            }
        except Exception as e:
            return {
                "success": False,
                "exit_code": -2,
                "output": f"ERROR: Execution failed: {str(e)}"
            }
print("✅ Security & Sandbox Guardrails Loaded!")

## 🤖 Step 4: Implement LLM Simulator (SimLLM)

In [ ]:
class SimLLM:
    def __init__(self, workspace_root: str):
        self.workspace_root = workspace_root

    def execute_child_card(self, deliverables: List[Dict[str, str]], iteration: int) -> Dict[str, Any]:
        logs = []
        created_files = []
        
        for deliv in deliverables:
            rel_path = deliv.get("path")
            path = os.path.join(self.workspace_root, rel_path)
            os.makedirs(os.path.dirname(path), exist_ok=True)
            
            if "test" in rel_path:
                code = """# Test suite for auth.py
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from app.auth import hash_password, verify_token

def test_hash_password():
    assert hash_password("secret") == "2bb80d537b1da3e38bd30361aa855686bde0eacd7162fef6a25fe97bf527a25b"

def test_verify_token():
    assert verify_token("valid-token") is True
    assert verify_token("invalid-token") is False

if __name__ == "__main__":
    test_hash_password()
    test_verify_token()
    print("All tests passed successfully!")
"""
                with open(path, 'w') as f: f.write(code)
                created_files.append(path)
                logs.append("Drafted test suite file tests/test_auth.py.")
                
            elif "auth" in rel_path:
                if iteration == 1:
                    code = """# JWT Authentication module - Iteration 1
import sys
import broken_library_that_does_not_exist

def hash_password(password: str) -> str:
    return password[::-1]

def verify_token(token: str) -> bool:
    return True
"""
                    logs.append("Drafted auth.py with intentional import error to test self-healing loop.")
                else:
                    code = """# JWT Authentication module - Corrected Iteration
import hashlib

def hash_password(password: str) -> str:
    return hashlib.sha256(password.encode()).hexdigest()

def verify_token(token: str) -> bool:
    if token == "invalid-token":
        return False
    return True
"""
                    logs.append("Corrected auth.py: Removed broken library import and updated password hashing.")
                with open(path, 'w') as f: f.write(code)
                created_files.append(path)
        return {
            "success": True,
            "logs": logs,
            "created_files": created_files
        }

    def generate_gap_analysis(self) -> List[Dict[str, Any]]:
        return [
            {
                "id": "child-004-jwt-refresh-tokens",
                "name": "Add JWT Refresh Token Support",
                "tactical_objective": "Extend app/auth.py to handle refresh token rotation.",
                "deliverables": [{"path": "app/auth.py"}],
                "test_commands": ["python tests/test_auth.py"]
            },
            {
                "id": "child-005-add-rate-limiting",
                "name": "Implement API Rate Limiting",
                "tactical_objective": "Create app/rate_limiter.py to block brute-force attempts.",
                "deliverables": [{"path": "app/rate_limiter.py"}],
                "test_commands": ["python tests/test_rate_limiter.py"]
            }
        ]
print("✅ LLM Simulator initialized successfully!")

## 🔄 Step 5: Implement the Orchestrator Loop

In [ ]:
class AutonomousEngine:
    def __init__(self, workspace: str):
        self.workspace = workspace

    def execute(self, mother_card_dict: Dict, child_card_dict: Dict) -> Dict[str, Any]:
        mother = MotherCard.from_dict(mother_card_dict)
        child = ChildCard.from_dict(child_card_dict)
        
        guard = PrivacyGuard(restricted_paths=mother.restricted_paths)
        sandbox = SandboxRunner(banned_commands=mother.banned_commands)
        llm = SimLLM(workspace_root=self.workspace)
        
        print(f"[START] Loaded Mother: '{mother.name}'")
        print(f"[START] Processing Child: '{child.name}'")
        
        # Privacy Guard Routing
        paths = [d.get("path") for d in child.deliverables]
        profile = guard.evaluate_routing_profile(paths, mother.privacy_mode)
        child.execution_profile_assigned = profile
        print(f"[SEC] Privacy evaluation: routing assigned is '{profile.upper()}'")
        
        audit_trail = []
        
        while child.current_iteration < child.max_iterations:
            child.current_iteration += 1
            print(f"\n  [RUN] Starting Iteration {child.current_iteration}/{child.max_iterations}...")
            
            # Generate code edits on disk
            res = llm.execute_child_card(child.deliverables, child.current_iteration)
            audit_trail.append({
                "iteration": child.current_iteration,
                "action": "code_edit",
                "details": f"LLM edited: {', '.join(res['created_files'])}"
            })
            
            # Sandbox Verification
            all_passed = True
            for cmd in child.validation_commands:
                print(f"  [TEST] Running Sandbox Command: '{cmd}'")
                run_res = sandbox.run_validation(cmd, cwd=self.workspace)
                
                audit_trail.append({
                    "iteration": child.current_iteration,
                    "action": "validation",
                    "cmd": cmd,
                    "exit_code": run_res["exit_code"],
                    "output": run_res["output"].strip()
                })
                
                if not run_res["success"]:
                    all_passed = False
                    print(f"  [WARN] Command Failed! Output:\n{run_res['output'].strip()}")
                    break
                else:
                    print("  [OK] Command Passed successfully!")
            
            if all_passed:
                child.phase = "Completed"
                print(f"\n[SUCCESS] Child Card successfully validated and completed in {child.current_iteration} iterations!")
                break
            else:
                print("  [INFO] Verification failed. Prompting self-healing script edits on next iteration.")
                
        if child.phase == "Completed":
            print("\n[INFO] Triggering Mother Gap-Analysis & Autonomous Discovery...")
            new_jobs = llm.generate_gap_analysis()
            for nj in new_jobs:
                print(f"  [NEW] Mother autonomously scheduled backlog successor: '{nj['name']}' ({nj['id']})")
                mother.backlog_queue.append(nj)
            mother.completed_children.append(child.id)
            
        return {
            "mother": mother,
            "child": child,
            "audit": audit_trail
        }
print("✅ Orchestration Daemon ready for execution!")

## 🧪 Step 6: Execute and Interact with the AJE Demo!

In [ ]:
# Define Mother Card in standard dict format
mother_spec = {
    "metadata": {
        "id": "mother-complaint-router",
        "name": "Customer Complaint Router Mother Card"
    },
    "spec": {
        "core_philosophy": "Strict typing, cryptography safety, robust rate limiting.",
        "safety_policies": {
            "privacy_mode": "hybrid",
            "restricted_paths": ["**/secrets.env", "**/database/credentials/*"],
            "banned_commands": ["rm -rf", "docker system prune"]
        }
    }
}

# Define Child Card
child_spec = {
    "metadata": {
        "id": "child-003-jwt-auth",
        "parent_mother_id": "mother-complaint-router",
        "name": "JWT Authentication Module"
    },
    "spec": {
        "tactical_objective": "Write app/auth.py with SHA256 password hashing.",
        "max_iterations": 5,
        "deliverables": [
            {"path": "app/auth.py"},
            {"path": "tests/test_auth.py"}
        ],
        "validation": {
            "test_commands": [
                "python tests/test_auth.py"
            ]
        }
    }
}

# Execute loop in notebook_workspace
engine = AutonomousEngine(workspace="notebook_workspace")
results = engine.execute(mother_spec, child_spec)

## 📑 Step 7: View the Forensic Audit Ledger

In [ ]:
print("=========================================================")
print("📜 IMMUTABLE FORENSIC AUDIT TRAILS")
print("=========================================================")
for idx, log in enumerate(results["audit"]):
    print(f"\n[{idx+1}] ITERATION {log['iteration']} | {log['action'].upper()}")
    if log["action"] == "code_edit":
        print(f"    Detail: {log['details']}")
    else:
        print(f"    Command Executed: {log['cmd']}")
        print(f"    Exit Code: {log['exit_code']}")
        print(f"    Sandbox Output:\n{log['output']}")

## 🧪 Step 8: Interactive Privacy Experiment!

Let's test what happens if a malicious or stray task tries to read confidential workspace directories. 

In the cell below, we will modify the Child Card deliverables to save database configurations inside `app/database/credentials/config.py` which is on the Mother Card's `restricted_paths` list.

Run the cell below and watch how the **Privacy Guard** automatically intercepts the request and routes the task locally to keep your private code secure.

In [ ]:
restricted_child_spec = {
    "metadata": {
        "id": "child-009-db-creds",
        "parent_mother_id": "mother-complaint-router",
        "name": "Database Config Integration"
    },
    "spec": {
        "tactical_objective": "Configure app/database/credentials/config.py with environment configurations.",
        "max_iterations": 5,
        "deliverables": [
            {"path": "app/database/credentials/config.py"},
            {"path": "tests/test_auth.py"}
        ],
        "validation": {
            "test_commands": [
                "python tests/test_auth.py"
            ]
        }
    }
}

print("--- RUNNING SECURE ISOLATION TEST ---")
secure_results = engine.execute(mother_spec, restricted_child_spec)